# Week 6 — Optional Extra: Deep Neural Network (Inference)

After the day-4 8-layer NN underperformed, this notebook runs a much deeper residual network (~289M parameters) trained on the full 800K dataset. Achieves $46.49 mean absolute error — competitive with frontier models.

Two notebooks for this:
- **redemption_train.ipynb** — train the network from scratch (~4 hours on M1 Mac GPU)
- **redemption_run.ipynb** (this one) — load pre-trained weights and run inference

**Before running:** drop the `deep_neural_network.pth` weights file into the `week6/` directory. If you haven't trained your own, train it via the redemption_train notebook first.

## Setup

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from pricer.deep_neural_network import DeepNeuralNetworkRunner
from pricer.items import Item

In [ ]:
LITE_MODE = False
load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

In [ ]:
username = "YOUR_HF_USERNAME"  # replace with your Hugging Face username
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

## Initialize the runner and load the trained weights

In [ ]:
runner = DeepNeuralNetworkRunner(train, val[:1000])
runner.setup()

In [ ]:
# Use the first line on Mac (with MPS / GPU); use the second on Windows / CPU
# runner.load('deep_neural_network.pth')
runner.load('deep_neural_network.pth', 'cpu')

## Run inference and evaluate

In [ ]:
def deep_neural_network(item):
    return runner.inference(item)

evaluate(deep_neural_network, test)